In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as subplots
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from scipy import stats
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

# Mobile Usage Analysis: Understanding Digital Behavior Patterns

## Challenge Overview
This analysis examines 3 months of anonymized mobile usage data to understand user digital behavior, identify productivity vs distraction patterns, and design intelligent intervention systems.

**Key Objectives:**
1. Analyze daily and weekly screen time trends across app categories
2. Measure productivity vs distraction balance
3. Understand YouTube engagement and content consumption
4. Segment users based on behavioral patterns
5. Detect anomalies and burnout indicators
6. Design smart recommendations and focus interventions

## 1. Data Loading and Exploration

**Approach:** Generate realistic mobile usage dataset with multiple users tracked over 90 days, capturing daily screen time across different app categories and YouTube engagement metrics.

In [2]:
# Load the dataset
df = pd.read_csv(r'Data Raw\screen_time_app_usage_dataset.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (3000, 24)

First few rows:
   user_id                           date     app_name       category  \
0     1051  2024-01-01 00:00:00.000000000       Camera      Utilities   
1     1088  2024-01-01 00:43:41.673891297       Chrome      Utilities   
2     1052  2024-01-01 01:27:23.347782594      Spotify  Entertainment   
3     1028  2024-01-01 02:11:05.021673891  Google Maps      Utilities   
4     1034  2024-01-01 02:54:46.695565188    Instagram         Social   

   screen_time_min  launches  interactions  is_productive  youtube_views  \
0            24.53         2             7          False            NaN   
1            19.78         3             2          False            NaN   
2            32.03         1             6          False            NaN   
3            19.10         2             5          False            NaN   
4            25.19         1             5          False            NaN   

   youtube_likes  ...  extra_col_14  extra_col_15  extra_col_

In [3]:
# Data Cleaning and Preprocessing
df['date'] = pd.to_datetime(df['date'])

# Drop unnecessary columns (extra_col_11 through extra_col_23)
df = df.drop(columns=[col for col in df.columns if col.startswith('extra_col')])

# Fill YouTube engagement metrics with 0 for non-YouTube apps
df['youtube_views'] = df['youtube_views'].fillna(0)
df['youtube_likes'] = df['youtube_likes'].fillna(0)
df['youtube_comments'] = df['youtube_comments'].fillna(0)

# Create useful features
df['week'] = df['date'].dt.isocalendar().week
df['day_of_week'] = df['date'].dt.day_name()
df['month'] = df['date'].dt.month
df['date_only'] = df['date'].dt.date

# Engagement ratio for YouTube
df['engagement_rate'] = df.apply(lambda x: (x['youtube_likes'] + x['youtube_comments']) / (x['youtube_views'] + 1), axis=1)

print("Cleaned Dataset Shape:", df.shape)
print("\nUnique Users:", df['user_id'].nunique())
print("Date Range:", df['date'].min(), "to", df['date'].max())
print("App Categories:", df['category'].unique())
print("Productivity Distribution:", df['is_productive'].value_counts())

Cleaned Dataset Shape: (3000, 16)

Unique Users: 101
Date Range: 2024-01-01 00:00:00 to 2024-04-01 00:00:00
App Categories: ['Utilities' 'Entertainment' 'Social' 'Productivity']
Productivity Distribution: is_productive
False    2247
True      753
Name: count, dtype: int64


In [4]:
# Exploratory Data Analysis - Screen Time Distribution by Category
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Screen Time by Category", "App Category Count", 
                   "Screen Time Distribution", "Productivity vs Non-Productive"),
    specs=[[{"type": "pie"}, {"type": "bar"}],
           [{"type": "box"}, {"type": "bar"}]]
)

# 1. Screen time by category (pie chart)
category_time = df.groupby('category')['screen_time_min'].sum()
fig.add_trace(
    go.Pie(labels=category_time.index, values=category_time.values, name="Category"),
    row=1, col=1
)

# 2. App category count (bar chart)
category_count = df['category'].value_counts()
fig.add_trace(
    go.Bar(x=category_count.index, y=category_count.values, name="Count", showlegend=False),
    row=1, col=2
)

# 3. Screen time distribution by category (box plot)
for category in df['category'].unique():
    category_data = df[df['category'] == category]['screen_time_min']
    fig.add_trace(
        go.Box(y=category_data, name=category, boxmean='sd'),
        row=2, col=1
    )

# 4. Productivity comparison (bar chart)
productivity_time = df.groupby('is_productive')['screen_time_min'].sum()
fig.add_trace(
    go.Bar(x=['Non-Productive', 'Productive'], y=productivity_time.values, 
           name="Total Time", showlegend=False),
    row=2, col=2
)

fig.update_layout(height=900, title_text="Screen Time Overview", showlegend=False)
fig.show()

print("Summary Statistics:")
print(df.groupby('category')['screen_time_min'].agg(['sum', 'mean', 'min', 'max']))

Summary Statistics:
                    sum       mean   min     max
category                                        
Entertainment  21858.71  29.984513  0.01  204.18
Productivity   22283.10  29.592430  0.01  218.39
Social         23753.28  31.131429  0.03  209.48
Utilities      21811.08  28.888848  0.01  185.65


## 2. Usage Pattern Analysis

Analyze daily and weekly screen time trends to identify behavioral cycles and peak usage hours.

In [5]:
# Daily Screen Time Trends
daily_time = df.groupby('date_only').agg({
    'screen_time_min': 'sum',
    'launches': 'sum',
    'interactions': 'sum'
}).reset_index()

daily_time['date_only'] = pd.to_datetime(daily_time['date_only'])

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("Daily Total Screen Time", "Daily Launches", "Daily Interactions"),
    shared_xaxes=True
)

fig.add_trace(go.Scatter(x=daily_time['date_only'], y=daily_time['screen_time_min'],
                         mode='lines+markers', name='Screen Time', line=dict(color='blue')),
              row=1, col=1)

fig.add_trace(go.Scatter(x=daily_time['date_only'], y=daily_time['launches'],
                         mode='lines+markers', name='Launches', line=dict(color='green')),
              row=2, col=1)

fig.add_trace(go.Scatter(x=daily_time['date_only'], y=daily_time['interactions'],
                         mode='lines+markers', name='Interactions', line=dict(color='orange')),
              row=3, col=1)

fig.update_xaxes(title_text="Date", row=3, col=1)
fig.update_yaxes(title_text="Minutes", row=1, col=1)
fig.update_yaxes(title_text="Count", row=2, col=1)
fig.update_yaxes(title_text="Count", row=3, col=1)
fig.update_layout(height=900, title_text="Daily Usage Trends Over 3 Months")
fig.show()

print("Daily Statistics:")
print(daily_time[['screen_time_min', 'launches', 'interactions']].describe())

Daily Statistics:
       screen_time_min  launches  interactions
count        92.000000  92.00000     92.000000
mean        975.067065  65.00000    161.804348
std         195.785073  11.04934     20.537416
min          18.500000   1.00000      2.000000
25%         841.540000  61.00000    154.750000
50%         989.790000  66.00000    164.000000
75%        1105.562500  71.25000    171.250000
max        1449.330000  91.00000    193.000000


In [6]:
# Weekly Category Comparison
weekly_category = df.groupby(['week', 'category'])['screen_time_min'].sum().reset_index()

fig = px.line(weekly_category, x='week', y='screen_time_min', color='category',
              title='Weekly Screen Time by Category',
              labels={'screen_time_min': 'Screen Time (minutes)', 'week': 'Week Number'},
              markers=True)
fig.show()

# Day-of-week analysis
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_category = df.groupby(['day_of_week', 'category'])['screen_time_min'].sum().reset_index()
daily_category['day_of_week'] = pd.Categorical(daily_category['day_of_week'], categories=day_order, ordered=True)
daily_category = daily_category.sort_values('day_of_week')

fig = px.bar(daily_category, x='day_of_week', y='screen_time_min', color='category',
             title='Average Screen Time by Day of Week',
             labels={'screen_time_min': 'Screen Time (minutes)', 'day_of_week': 'Day of Week'},
             barmode='stack')
fig.show()

print("\nDay-of-Week Summary:")
print(df.groupby('day_of_week')['screen_time_min'].agg(['sum', 'mean']))


Day-of-Week Summary:
                  sum       mean
day_of_week                     
Friday       12900.56  30.071235
Monday       12779.40  29.719535
Saturday     13493.61  31.527126
Sunday       12372.77  28.908341
Thursday     12498.16  29.201308
Tuesday      12909.62  30.162664
Wednesday    12752.05  29.725058


## 3. Productivity vs Distraction Classification

Analyze the balance between productive and non-productive app usage, and identify high-distraction user profiles.

In [7]:
# User-Level Productivity Analysis
user_metrics = df.groupby('user_id').agg({
    'screen_time_min': 'sum',
    'launches': 'sum',
    'interactions': 'sum',
    'is_productive': lambda x: (x.sum() / len(x)) * 100  # Productivity percentage
}).rename(columns={'is_productive': 'productivity_ratio'}).reset_index()

# Calculate distraction score and category distribution
user_category_time = df.groupby(['user_id', 'category'])['screen_time_min'].sum().unstack(fill_value=0)
user_category_time['total_time'] = user_category_time.sum(axis=1)

# Calculate ratios
for col in user_category_time.columns[:-1]:
    user_category_time[f'{col}_ratio'] = (user_category_time[col] / user_category_time['total_time']) * 100

# Merge with main user metrics
user_metrics = user_metrics.merge(user_category_time[['total_time', 'Entertainment_ratio', 'Social_ratio', 'Productivity_ratio']], 
                                  left_on='user_id', right_index=True)

# Identify user segments
def segment_user(row):
    if row['Productivity_ratio'] > 40:
        return 'Highly Productive'
    elif row['Entertainment_ratio'] > 35:
        return 'Entertainment-Heavy'
    elif row['Social_ratio'] > 35:
        return 'Social-First'
    else:
        return 'Balanced'

user_metrics['segment'] = user_metrics.apply(segment_user, axis=1)

print("User Productivity Metrics:")
print(user_metrics[['user_id', 'productivity_ratio', 'segment']].head(20))

# Visualization: Productivity Distribution
fig = go.Figure()
fig.add_trace(go.Histogram(x=user_metrics['productivity_ratio'], nbinsx=30, name='Productivity Score'))
fig.update_layout(title='Distribution of User Productivity Ratios',
                  xaxis_title='Productivity Ratio (%)',
                  yaxis_title='Number of Users')
fig.show()

# User Segment Distribution
segment_counts = user_metrics['segment'].value_counts()
fig = px.pie(values=segment_counts.values, names=segment_counts.index, 
             title='User Segmentation Distribution')
fig.show()

print("\nSegment Summary:")
print(user_metrics.groupby('segment').agg({
    'productivity_ratio': 'mean',
    'screen_time_min': 'mean',
    'Entertainment_ratio': 'mean',
    'Social_ratio': 'mean'
}).round(2))

User Productivity Metrics:
    user_id  productivity_ratio              segment
0      1000           24.137931             Balanced
1      1001           30.555556         Social-First
2      1002           20.000000             Balanced
3      1003           25.000000             Balanced
4      1004           26.666667  Entertainment-Heavy
5      1005           36.363636  Entertainment-Heavy
6      1006           15.151515             Balanced
7      1007           27.027027  Entertainment-Heavy
8      1008           27.586207         Social-First
9      1009           26.470588         Social-First
10     1010           32.432432             Balanced
11     1011           20.689655             Balanced
12     1012           28.000000         Social-First
13     1013           24.242424         Social-First
14     1014           33.333333    Highly Productive
15     1015           11.111111  Entertainment-Heavy
16     1016           40.000000    Highly Productive
17     1017        


Segment Summary:
                     productivity_ratio  screen_time_min  Entertainment_ratio  \
segment                                                                         
Balanced                          24.80           885.56                23.85   
Entertainment-Heavy               23.18           960.89                40.36   
Highly Productive                 36.65           841.55                16.54   
Social-First                      22.08           871.50                17.35   

                     Social_ratio  
segment                            
Balanced                    24.01  
Entertainment-Heavy         17.47  
Highly Productive           17.10  
Social-First                42.56  


## 4. YouTube Engagement Analysis

Analyze relationships between screen time on YouTube and engagement metrics (views, likes, comments).

In [8]:
# YouTube Engagement Analysis
youtube_data = df[df['app_name'] == 'YouTube'].copy()
youtube_data = youtube_data[youtube_data['youtube_views'] > 0]

print(f"YouTube Sessions with engagement data: {len(youtube_data)}")
print("\nYouTube Engagement Statistics:")
print(youtube_data[['screen_time_min', 'youtube_views', 'youtube_likes', 'youtube_comments']].describe())

# Correlation Analysis
if len(youtube_data) > 0:
    correlation_vars = ['screen_time_min', 'youtube_views', 'youtube_likes', 'youtube_comments']
    corr_matrix = youtube_data[correlation_vars].corr()
    
    fig = px.imshow(corr_matrix, text_auto=True, aspect="auto", 
                    title='Correlation: Screen Time vs YouTube Engagement',
                    labels=dict(color='Correlation'))
    fig.show()
    
    # Scatter plots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("Time vs Views", "Time vs Likes", "Time vs Comments", "Views vs Engagement"),
        specs=[[{"type": "scatter"}, {"type": "scatter"}],
               [{"type": "scatter"}, {"type": "scatter"}]]
    )
    
    fig.add_trace(
        go.Scatter(x=youtube_data['screen_time_min'], y=youtube_data['youtube_views'],
                   mode='markers', name='Time vs Views'),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=youtube_data['screen_time_min'], y=youtube_data['youtube_likes'],
                   mode='markers', name='Time vs Likes'),
        row=1, col=2
    )
    
    fig.add_trace(
        go.Scatter(x=youtube_data['screen_time_min'], y=youtube_data['youtube_comments'],
                   mode='markers', name='Time vs Comments'),
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=youtube_data['youtube_views'], 
                   y=youtube_data['youtube_likes'] + youtube_data['youtube_comments'],
                   mode='markers', name='Views vs Engagement'),
        row=2, col=2
    )
    
    fig.update_layout(height=800, title_text="YouTube Engagement Analysis")
    fig.show()
    
    # User-level YouTube consumption
    user_youtube = youtube_data.groupby('user_id').agg({
        'screen_time_min': 'sum',
        'youtube_views': 'sum',
        'youtube_likes': 'sum',
        'youtube_comments': 'sum'
    }).reset_index()
    
    user_youtube['engagement_rate'] = (user_youtube['youtube_likes'] + user_youtube['youtube_comments']) / (user_youtube['youtube_views'] + 1)
    user_youtube = user_youtube.sort_values('youtube_views', ascending=False).head(20)
    
    print("\nTop 20 YouTube Consumers:")
    print(user_youtube)

YouTube Sessions with engagement data: 142

YouTube Engagement Statistics:
       screen_time_min  youtube_views  youtube_likes  youtube_comments
count       142.000000     142.000000     142.000000        142.000000
mean         33.384507  256577.183099   14645.267606       2695.563380
std          34.246342  135872.940749   10511.930022       2147.688995
min           0.480000    1738.000000      69.000000          8.000000
25%           7.595000  141490.000000    6548.250000        957.750000
50%          22.125000  249747.000000   12928.500000       2325.500000
75%          47.395000  364498.000000   19929.000000       3671.500000
max         204.180000  494839.000000   43893.000000       8934.000000



Top 20 YouTube Consumers:
    user_id  screen_time_min  youtube_views  youtube_likes  youtube_comments  \
58     1072            30.86      1592701.0        76714.0           17590.0   
74     1091           212.54      1562654.0        59672.0           14539.0   
70     1085           102.34      1324141.0        83853.0           13017.0   
80     1100           177.59      1051551.0        49034.0           12180.0   
1      1002           145.89       874819.0        39093.0           11001.0   
22     1028            80.67       845612.0        76265.0            8170.0   
76     1093           129.54       845230.0        23747.0           10801.0   
6      1007            40.06       824677.0        68744.0            3431.0   
51     1064           134.21       817583.0        73571.0            6489.0   
34     1041            76.07       817210.0        28233.0           13422.0   
10     1013            37.07       762801.0        49886.0            6032.0   
20     1026  

## 5. User Behavioral Segmentation (K-Means Clustering)

Apply clustering to identify distinct user behavioral groups.

In [9]:
# Prepare features for clustering
clustering_features = user_category_time[[col for col in user_category_time.columns if col.endswith('_ratio')]].fillna(0)

# Standardize features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(clustering_features)

# Elbow method to find optimal clusters
inertias = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(scaled_features)
    inertias.append(kmeans.inertia_)

fig = px.line(x=range(2, 11), y=inertias, markers=True,
              title='Elbow Method For Optimal K',
              labels={'x': 'Number of Clusters', 'y': 'Inertia'})
fig.show()

# Apply K-Means with optimal clusters (let's use k=4)
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
user_metrics['cluster'] = kmeans.fit_predict(scaled_features)

# PCA for visualization
pca = PCA(n_components=2)
pca_features = pca.fit_transform(scaled_features)

fig = px.scatter(x=pca_features[:, 0], y=pca_features[:, 1], 
                 color=user_metrics['cluster'].astype(str),
                 title=f'User Clusters (PCA Projection) - K={optimal_k}',
                 labels={'x': f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
                        'y': f'PC2 ({pca.explained_variance_ratio_[1]:.1%})'},
                 hover_data={'user_id': user_metrics['user_id']})
fig.show()

# Cluster profiles
print("\nCluster Profiles:")
for cluster_id in range(optimal_k):
    cluster_data = user_metrics[user_metrics['cluster'] == cluster_id]
    print(f"\n--- Cluster {cluster_id} ({len(cluster_data)} users) ---")
    print(f"Avg Productivity Ratio: {cluster_data['productivity_ratio'].mean():.1f}%")
    print(f"Avg Screen Time: {cluster_data['screen_time_min'].mean():.0f} min")
    print(f"Avg Entertainment Time: {cluster_data['Entertainment_ratio'].mean():.1f}%")
    print(f"Avg Social Time: {cluster_data['Social_ratio'].mean():.1f}%")
    
    # Characterize the cluster
    if cluster_data['productivity_ratio'].mean() > 40:
        print("Profile: HIGHLY PRODUCTIVE")
    elif cluster_data['Entertainment_ratio'].mean() > 40:
        print("Profile: ENTERTAINMENT-HEAVY")
    elif cluster_data['Social_ratio'].mean() > 40:
        print("Profile: SOCIAL-FIRST")
    else:
        print("Profile: BALANCED")


Cluster Profiles:

--- Cluster 0 (35 users) ---
Avg Productivity Ratio: 24.8%
Avg Screen Time: 930 min
Avg Entertainment Time: 34.9%
Avg Social Time: 22.4%
Profile: BALANCED

--- Cluster 1 (17 users) ---
Avg Productivity Ratio: 35.1%
Avg Screen Time: 854 min
Avg Entertainment Time: 18.7%
Avg Social Time: 14.6%
Profile: BALANCED

--- Cluster 2 (18 users) ---
Avg Productivity Ratio: 21.8%
Avg Screen Time: 884 min
Avg Entertainment Time: 17.4%
Avg Social Time: 22.6%
Profile: BALANCED

--- Cluster 3 (31 users) ---
Avg Productivity Ratio: 22.6%
Avg Screen Time: 863 min
Avg Entertainment Time: 18.0%
Avg Social Time: 40.4%
Profile: SOCIAL-FIRST


## 6. Anomaly & Trend Detection

Detect unusual spikes in screen time and identify potential burnout periods.

In [10]:
# Anomaly Detection using Isolation Forest
anomaly_features = daily_time[['screen_time_min', 'launches', 'interactions']].values
iso_forest = IsolationForest(contamination=0.05, random_state=42)
daily_time['anomaly'] = iso_forest.fit_predict(anomaly_features)
daily_time['is_anomaly'] = daily_time['anomaly'] == -1

# Visualize anomalies
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_time['date_only'], 
    y=daily_time['screen_time_min'],
    mode='lines+markers',
    name='Normal',
    marker=dict(size=8, color='blue')
))

anomalies = daily_time[daily_time['is_anomaly']]
fig.add_trace(go.Scatter(
    x=anomalies['date_only'],
    y=anomalies['screen_time_min'],
    mode='markers',
    name='Anomaly',
    marker=dict(size=12, color='red', symbol='x')
))

fig.update_layout(title='Daily Screen Time with Anomalies Detected',
                  xaxis_title='Date',
                  yaxis_title='Screen Time (minutes)')
fig.show()

print(f"\nTotal anomalies detected: {daily_time['is_anomaly'].sum()}")
print("\nAnomalous Days (High Screen Time):")
print(daily_time[daily_time['is_anomaly']][['date_only', 'screen_time_min', 'launches', 'interactions']])

# Per-user trend analysis
user_daily = df.groupby(['user_id', 'date_only']).agg({
    'screen_time_min': 'sum',
    'is_productive': 'mean'
}).reset_index()

# Detect potential burnout (sudden decreases after high usage)
print("\n\nUsers with Potential Burnout Patterns:")
burnout_users = []

for user_id in user_daily['user_id'].unique():
    user_data = user_daily[user_daily['user_id'] == user_id].sort_values('date_only')
    if len(user_data) > 7:
        # Calculate rolling average
        user_data['rolling_avg'] = user_data['screen_time_min'].rolling(window=7).mean()
        # Detect sudden drops (potential burnout recovery)
        drops = user_data['screen_time_min'].diff() < -100
        if drops.sum() > 0:
            burnout_users.append({
                'user_id': user_id,
                'max_screen_time': user_data['screen_time_min'].max(),
                'drops_detected': drops.sum()
            })

if burnout_users:
    burnout_df = pd.DataFrame(burnout_users).sort_values('max_screen_time', ascending=False).head(10)
    print(burnout_df)


Total anomalies detected: 5

Anomalous Days (High Screen Time):
    date_only  screen_time_min  launches  interactions
9  2024-01-10          1449.33        68           168
17 2024-01-18           678.63        45           145
26 2024-01-27           845.86        48           193
44 2024-02-14           761.12        91           155
91 2024-04-01            18.50         1             2


Users with Potential Burnout Patterns:
    user_id  max_screen_time  drops_detected
24     1042           324.06               2
29     1054           321.37               1
16     1031           243.01               2
21     1038           229.63               2
2      1003           228.43               3
32     1059           218.39               3
33     1060           217.16               3
37     1072           215.94               1
41     1083           214.45               1
39     1080           212.87               2


## 7. Recommendation & Intervention Design

Design intelligent recommendations and focus-assist features based on user behavior patterns.

In [11]:
# Recommendation Engine
def generate_recommendations(user_id, user_metrics, df):
    """Generate personalized recommendations for a user"""
    user_data = user_metrics[user_metrics['user_id'] == user_id].iloc[0]
    recommendations = []
    
    productivity_ratio = user_data['productivity_ratio']
    entertainment_ratio = user_data['Entertainment_ratio']
    social_ratio = user_data['Social_ratio']
    total_time = user_data['screen_time_min']
    
    # Rule 1: Low productivity warning
    if productivity_ratio < 20:
        recommendations.append({
            'type': 'PRODUCTIVITY_ALERT',
            'severity': 'HIGH',
            'message': 'Your productivity app usage is very low. Consider scheduling focus blocks.',
            'action': 'Enable Focus Mode during 9-12 AM and 2-5 PM'
        })
    
    # Rule 2: High entertainment content
    if entertainment_ratio > 40:
        recommendations.append({
            'type': 'ENTERTAINMENT_LIMIT',
            'severity': 'MEDIUM',
            'message': 'Entertainment apps consume over 40% of your time.',
            'action': 'Set app limits on Entertainment category: 1 hour daily max'
        })
    
    # Rule 3: High social media
    if social_ratio > 35:
        recommendations.append({
            'type': 'SOCIAL_ALERT',
            'severity': 'MEDIUM',
            'message': 'Social media accounts for significant screen time.',
            'action': 'Limit social apps to 30 min per session with 2-hour cooldown'
        })
    
    # Rule 4: Total screen time
    if total_time > 3000:  # Assuming monthly data
        recommendations.append({
            'type': 'USAGE_ALERT',
            'severity': 'HIGH',
            'message': 'Total screen time is very high.',
            'action': 'Target 20% reduction. Use app blockers and digital sunset (no phones after 9 PM)'
        })
    elif total_time > 2000:
        recommendations.append({
            'type': 'USAGE_MODERATE',
            'severity': 'LOW',
            'message': 'Screen time is moderate - good balance.',
            'action': 'Continue current habits with minor optimizations'
        })
    
    # Rule 5: Focus period suggestion
    if productivity_ratio < 30:
        recommendations.append({
            'type': 'FOCUS_SUGGESTION',
            'severity': 'MEDIUM',
            'message': 'Establish dedicated focus periods.',
            'action': 'Focus Assist: 8-10 AM (morning peak) and 2-3 PM (afternoon dip)'
        })
    
    return recommendations

# Generate recommendations for sample users
print("=" * 80)
print("PERSONALIZED RECOMMENDATIONS FOR HIGH-RISK USERS")
print("=" * 80)

high_risk = user_metrics[user_metrics['productivity_ratio'] < 25].head(5)

for idx, user in high_risk.iterrows():
    print(f"\n{'='*80}")
    print(f"USER ID: {user['user_id']} | Productivity: {user['productivity_ratio']:.1f}% | Cluster: {user['cluster']}")
    print(f"{'='*80}")
    
    recommendations = generate_recommendations(user['user_id'], user_metrics, df)
    
    for i, rec in enumerate(recommendations, 1):
        print(f"\n{i}. [{rec['severity']}] {rec['type']}")
        print(f"   Message: {rec['message']}")
        print(f"   Action:  {rec['action']}")

# Summary statistics on interventions needed
need_intervention = user_metrics[
    (user_metrics['productivity_ratio'] < 25) | 
    (user_metrics['Entertainment_ratio'] > 45) |
    (user_metrics['screen_time_min'] > 3000)
]

print(f"\n\n{'='*80}")
print("INTERVENTION SUMMARY")
print(f"{'='*80}")
print(f"Total users: {len(user_metrics)}")
print(f"Users needing intervention: {len(need_intervention)} ({len(need_intervention)/len(user_metrics)*100:.1f}%)")
print(f"\nRisk Categories:")
print(f"  - Low Productivity (<25%): {(user_metrics['productivity_ratio'] < 25).sum()} users")
print(f"  - High Entertainment (>45%): {(user_metrics['Entertainment_ratio'] > 45).sum()} users")
print(f"  - High Social Media (>40%): {(user_metrics['Social_ratio'] > 40).sum()} users")
print(f"  - Excessive Screen Time (>3000 min): {(user_metrics['screen_time_min'] > 3000).sum()} users")

PERSONALIZED RECOMMENDATIONS FOR HIGH-RISK USERS

USER ID: 1000 | Productivity: 24.1% | Cluster: 3

1. [MEDIUM] FOCUS_SUGGESTION
   Message: Establish dedicated focus periods.
   Action:  Focus Assist: 8-10 AM (morning peak) and 2-3 PM (afternoon dip)

USER ID: 1002 | Productivity: 20.0% | Cluster: 0

1. [MEDIUM] FOCUS_SUGGESTION
   Message: Establish dedicated focus periods.
   Action:  Focus Assist: 8-10 AM (morning peak) and 2-3 PM (afternoon dip)

USER ID: 1006 | Productivity: 15.2% | Cluster: 2

1. [HIGH] PRODUCTIVITY_ALERT
   Message: Your productivity app usage is very low. Consider scheduling focus blocks.
   Action:  Enable Focus Mode during 9-12 AM and 2-5 PM

2. [MEDIUM] FOCUS_SUGGESTION
   Message: Establish dedicated focus periods.
   Action:  Focus Assist: 8-10 AM (morning peak) and 2-3 PM (afternoon dip)

USER ID: 1011 | Productivity: 20.7% | Cluster: 0

1. [MEDIUM] FOCUS_SUGGESTION
   Message: Establish dedicated focus periods.
   Action:  Focus Assist: 8-10 AM (morning